In [ ]:
# Setup and Imports
!pip install -q transformers[torch] datasets accelerate

import os
import json
import pandas as pd
import numpy as np
import torch
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.metrics import f1_score, r2_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EvalPrediction,
    RobertaPreTrainedModel, RobertaModel
)
from torch.nn import MSELoss
from datasets import Dataset
from google.colab import drive
from transformers.trainer_utils import get_last_checkpoint

print("Mounting Google Drive...")
drive.mount('/content/drive')

GDRIVE_PATH = '/content/drive/MyDrive/'
MODELS_DIR = os.path.join(GDRIVE_PATH, 'LargeModelsFinal/')
RESULTS_DIR = os.path.join(GDRIVE_PATH, 'LargeModels_Results/')
CHECKPOINTS_DIR = os.path.join(GDRIVE_PATH, 'LargeModels_Checkpoints/')
DATA_DIR = os.path.join(GDRIVE_PATH, 'FinalData/')
SEED = 42
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

In [ ]:
# Helper class and Functions
class HF_Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels): self.encodings, self.labels = encodings, labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item
    def __len__(self): return len(self.labels)

def compute_clf_metrics(p: EvalPrediction):
    preds = torch.sigmoid(torch.from_numpy(p.predictions)).numpy() > 0.5
    f1 = f1_score(p.label_ids, preds, average='weighted', zero_division=0)
    return {'f1_weighted': f1}

def compute_reg_metrics(p: EvalPrediction):
    r2 = r2_score(p.label_ids, p.predictions)
    return {'r2_score': r2}

class RobertaForRegression(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids, attention_mask=attention_mask, **kwargs)
        cls_token_output = outputs[0][:, 0, :]
        logits = self.classifier(cls_token_output)
        preds = torch.sigmoid(logits)
        loss = None
        if labels is not None:
            loss_fct = MSELoss()
            loss = loss_fct(preds.squeeze(), labels.squeeze())
        return {"loss": loss, "logits": preds}

print(" Helper classes and functions defined.")


In [ ]:
# Master Training Function
def train_large_transformer(dataset_name, df, task_type, text_col, label_cols, model_name="roberta-large"):
    print(f"\n--- Starting Transformer Process for: {dataset_name} with {model_name} ---")

    model_folder = os.path.join(MODELS_DIR, dataset_name)
    metrics_folder = os.path.join(RESULTS_DIR, dataset_name)
    os.makedirs(metrics_folder, exist_ok=True)
    os.makedirs(model_folder, exist_ok=True)

    train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)
    train_dataset_hf = Dataset.from_pandas(train_df)
    test_dataset_hf = Dataset.from_pandas(test_df)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_and_format(examples):
        tokenized = tokenizer(examples[text_col], truncation=True, padding="max_length", max_length=128)
        tokenized["labels"] = [list(row) for row in zip(*[examples[col] for col in label_cols])]
        return tokenized

    print("Tokenizing data efficiently...")
    train_dataset = train_dataset_hf.map(tokenize_and_format, batched=True, remove_columns=train_df.columns.tolist())
    test_dataset = test_dataset_hf.map(tokenize_and_format, batched=True, remove_columns=test_df.columns.tolist())

    if task_type == 'classification':
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_cols), problem_type="multi_label_classification")
        compute_metrics = compute_clf_metrics
    else:
        model = RobertaForRegression.from_pretrained(model_name, num_labels=len(label_cols))
        compute_metrics = compute_reg_metrics

    args = TrainingArguments(
        output_dir=os.path.join(CHECKPOINTS_DIR, dataset_name),
        num_train_epochs=3,
        learning_rate=1e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        fp16=True,
        report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=test_dataset, compute_metrics=compute_metrics)

    last_checkpoint = get_last_checkpoint(args.output_dir)
    print(f" Starting training for {dataset_name}...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
    print(" Training finished. Evaluating...")
    eval_results = trainer.evaluate()

    trainer.save_model(model_folder)
    print(f" Model saved to: {model_folder}")

    results_path = os.path.join(metrics_folder, 'transformer_metrics.json')
    with open(results_path, 'w') as f: json.dump(eval_results, f, indent=4)
    print(f" Metrics saved to: {results_path}")

print(" Master transformer training function defined.")

In [ ]:
# EssaysBig5
print("\n--- Training RoBERTa-large for Essaysbig5 ---")
df_essays = pd.read_csv(os.path.join(DATA_DIR, 'essaysbig5_clean.csv')).dropna()
train_large_transformer('Essaysbig5', df_essays, 'classification', 'text', ['O', 'C', 'E', 'A', 'N'])


In [ ]:
# GoEmotions
print("\n--- Training RoBERTa-large for GoEmotions ---")
from sklearn.preprocessing import MultiLabelBinarizer
df_go = pd.read_csv(os.path.join(DATA_DIR, 'goemotions_clean.csv')).dropna()
mlb = MultiLabelBinarizer()
y_go_labels = mlb.fit_transform(df_go['emotions'].str.split(',')).astype('float32')
df_go_transformed = pd.concat([df_go.reset_index(drop=True), pd.DataFrame(y_go_labels, columns=mlb.classes_)], axis=1)
label_cols_go = mlb.classes_.tolist()
train_large_transformer('GoEmotions', df_go_transformed, 'classification', 'text', label_cols_go)


In [ ]:
# Pandora
print("\n--- Training RoBERTa-large for Pandora ---")
from sklearn.preprocessing import MinMaxScaler
df_pandora = pd.read_csv(os.path.join(DATA_DIR, 'pandora_clean.csv')).dropna()
label_cols_pandora = ['agreeableness', 'openness', 'conscientiousness', 'extraversion', 'neuroticism']
scaler = MinMaxScaler()
df_pandora_scaled = df_pandora.copy()
df_pandora_scaled[label_cols_pandora] = scaler.fit_transform(df_pandora[label_cols_pandora])
model_save_dir = os.path.join(MODELS_DIR, 'Pandora')
os.makedirs(model_save_dir, exist_ok=True)
joblib.dump(scaler, os.path.join(model_save_dir, 'scaler_pandora.pkl'))
train_large_transformer('Pandora', df_pandora_scaled, 'regression', 'text', label_cols_pandora)


In [ ]:
# EmoBank
print("\n--- Training RoBERTa-large for EmoBank ---")
from sklearn.preprocessing import MinMaxScaler
df_emobank = pd.read_csv(os.path.join(DATA_DIR, 'emobank_clean.csv')).dropna()
label_cols_emobank = ['V', 'A', 'D']
scaler = MinMaxScaler()
df_emobank_scaled = df_emobank.copy()
df_emobank_scaled[label_cols_emobank] = scaler.fit_transform(df_emobank[label_cols_emobank])
model_save_dir = os.path.join(MODELS_DIR, 'EmoBank')
os.makedirs(model_save_dir, exist_ok=True)
joblib.dump(scaler, os.path.join(model_save_dir, 'scaler_emobank.pkl'))
train_large_transformer('EmoBank', df_emobank_scaled, 'regression', 'text', label_cols_emobank)
